In [ ]:
# ============================================================
# Explainable AI for Credit Risk: German Dataset
# ============================================================
#
# This notebook contains the German dataset analysis used in the MSc
# dissertation "Explainable AI for Credit Risk Assessment".
#
# The workflow covers data preparation, four predictive models, SHAP, LIME
# and Integrated Gradients explanations, demographic outcome analysis, and
# the TensorFlow/PyTorch reproducibility comparison.
#
# Categorical feature attributions are aggregated back to their original
# parent features before ranking so that explanation methods can be compared
# on a common feature representation. Helper functions are also used to
# standardise the different output structures returned by the SHAP explainers.
#
# The German dataset encodes sex within the combined personal-status field.
# This is decoded according to the dataset definitions before being used in
# the demographic outcome analysis.
#
# For the TensorFlow/PyTorch comparison, common data splits and fixed
# explanation samples are used to keep the two implementations as closely
# matched as practicable. Repeated executions may nevertheless produce small
# numerical differences because complete determinism is not guaranteed,
# particularly for TensorFlow and stochastic LIME sampling.
#
# Integrated Gradients is implemented with Captum for PyTorch and directly
# with TensorFlow automatic differentiation using the method described by
# Sundararajan, Taly and Yan (2017).


In [ ]:
# ============================================================
# Setup (Google Colab)
# ============================================================

# Google Colab provides the core scientific Python and machine-learning libraries.
# Install the additional packages required for dataset access and XAI methods.
!pip install -q ucimlrepo
!pip install -q shap lime captum

# --- Standard Library ---
import time, copy

# --- Core Numerics and Data ---
import numpy as np
import pandas as pd

# --- Dataset Source ---
from ucimlrepo import fetch_ucirepo

# --- Models ---
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow import keras
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib
import matplotlib.pyplot as plt

# --- Preprocessing, Model Selection, Metrics ---
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

# --- Explanation Methods ---
import shap
import lime.lime_tabular
from captum.attr import IntegratedGradients

# --- Statistical Tests ---
from scipy.stats import spearmanr, wilcoxon

# --- Record Key Library Versions ---
# Record the main package versions used in the Colab environment
import sklearn, xgboost
print("tensorflow", tf.__version__)
print("torch", torch.__version__)
print("sklearn", sklearn.__version__)
print("xgboost", xgboost.__version__)

In [ ]:
# ============================================================
# Load the German Data
# ============================================================
# Data: Hofmann (1994), Statlog German Credit, UCI Machine Learning Repository

# Fetch the Statlog German Credit dataset directly from the UCI repository (dataset ID 144)
german = fetch_ucirepo(id=144)

X_german = german.data.features
y_german = german.data.targets

# Replace the generic attribute labels with the descriptive names provided by UCI
readable = german.variables.set_index('name')['description'].to_dict()
X_german = X_german.rename(columns=readable)

X_german.columns.tolist()

In [ ]:
# ============================================================
# Recode the Target
# ============================================================

# Recode the original classes (1 = good, 2 = bad) so that 1 represents
# the adverse outcome, consistent with the Taiwan default indicator
y_german = y_german.squeeze()
y_german = (y_german == 2).astype(int)

# Check the class distribution; approximately 30% of observations belong to the adverse class
print(y_german.value_counts(normalize=True).round(3))

In [ ]:
# ============================================================
# Decode Sex from the Personal-Status Field (Needed for RQ3)
# ============================================================

# The German dataset combines sex and marital status in a single field.
# Decode sex using the dataset definitions: A91/A93/A94 = male, A92/A95 = female
sex_map = {'A91': 'male', 'A92': 'female', 'A93': 'male', 'A94': 'male', 'A95': 'female'}
X_german['SEX'] = X_german['Personal status and sex'].map(sex_map)

print(X_german['SEX'].value_counts())

In [ ]:
# ============================================================
# First Look at the Data
# ============================================================

# Inspect the class distribution, demographic variables and feature cardinalities
# before modelling to confirm the recoding and overall dataset structure
print("Target balance:")
print(y_german.value_counts(normalize=True).round(3))
print()
print("SEX:")
print(X_german['SEX'].value_counts())
print()
print("Foreign worker:")
print(X_german['foreign worker'].value_counts())
print()
print("Age:")
print(X_german['Age'].describe())
print()
print("Distinct values per column:")
print(X_german.nunique().sort_values())

In [ ]:
# ============================================================
# Prepare Features
# ============================================================

# Remove the combined personal-status field after extracting sex to avoid duplicate information
X_german = X_german.drop(columns=['Personal status and sex'])

# Encode sex as a single binary indicator: 0 = male, 1 = female
# This variable is retained for the demographic outcome analysis
X_german['SEX'] = X_german['SEX'].map({'male': 0, 'female': 1})

In [ ]:
# ============================================================
# Train / Test Split
# ============================================================

# Split before encoding and scaling to prevent information from the test set entering preprocessing
# Stratify to preserve the approximately 30% adverse-outcome rate, using a fixed random seed
X_train, X_test, y_train, y_test = train_test_split(
    X_german, y_german, test_size=0.2, stratify=y_german, random_state=42)
print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True).round(3))

In [ ]:
# ============================================================
# Encode and Scale (Fit on Train Only)
# ============================================================

# Work on explicit copies of the training and test sets
X_train = X_train.copy()
X_test = X_test.copy()

# Treat the coded categorical variables as nominal and one-hot encode them
# SEX is retained as a binary indicator and is therefore excluded from scaling
cat_cols = X_train.select_dtypes(include='object').columns.tolist()
numeric_cols = [c for c in X_train.select_dtypes(include='number').columns if c != 'SEX']

# One-hot encode the categorical variables, dropping the reference level
X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True, dtype=int)
X_test  = pd.get_dummies(X_test,  columns=cat_cols, drop_first=True, dtype=int)

# Align the test-set columns with the training set in case a category is absent from one split
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Standardise numeric features using parameters fitted on the training set only
scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])

print(X_train.shape, X_test.shape)
X_train.head()

In [ ]:
# ============================================================
# Information Value (Diagnostic, No Features Dropped)
# ============================================================

# Use Information Value as a diagnostic only; no features are removed on this basis
# Calculate it on the unencoded training data so each original feature is assessed once
X_iv = X_german.loc[X_train.index].copy()
y_iv = y_train

def iv_for_feature(x, y, n_bins=10):
    # Use quantile bins for variables with many distinct values; otherwise retain the observed categories
    if x.nunique() > 20:
        binned = pd.qcut(x, n_bins, duplicates='drop')
    else:
        binned = x.astype('category')

    d = pd.DataFrame({'bin': binned, 'y': y.values})
    grp = d.groupby('bin', observed=True)['y'].agg(['count', 'sum'])
    grp.columns = ['total', 'bad']          # bad = the adverse outcome (y = 1)
    grp['good'] = grp['total'] - grp['bad']

    # Apply 0.5 smoothing to avoid infinite WoE values when a bin contains no good or adverse outcomes
    n = len(grp)
    grp['dist_bad']  = (grp['bad']  + 0.5) / (grp['bad'].sum()  + 0.5 * n)
    grp['dist_good'] = (grp['good'] + 0.5) / (grp['good'].sum() + 0.5 * n)

    grp['woe'] = np.log(grp['dist_good'] / grp['dist_bad'])
    return ((grp['dist_good'] - grp['dist_bad']) * grp['woe']).sum()

# Retain all features regardless of their IV score because the demographic variables
# are required for fairness analysis and IV does not capture multivariate interactions
iv_table = pd.Series(
    {col: iv_for_feature(X_iv[col], y_iv) for col in X_iv.columns}
).sort_values(ascending=False).round(4)

print(iv_table)

In [ ]:
# ============================================================
# Model 1: Logistic Regression
# ============================================================
# Use class weighting rather than synthetic resampling to address class imbalance
# while retaining the original observations for explanation and fairness analysis

# Use logistic regression as the transparent baseline model
# Balanced class weights are applied without resampling the training observations
logreg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
logreg.fit(X_train, y_train)

proba = logreg.predict_proba(X_test)[:, 1]
print("Test AUC:", round(roc_auc_score(y_test, proba), 4))
print()
print(classification_report(y_test, logreg.predict(X_test)))

# Use the same fixed cross-validation folds across all models for consistent comparison
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(logreg, X_train, y_train, cv=cv, scoring='roc_auc')
print("CV AUC (5-fold):", cv_auc.round(4))
print("Mean CV AUC:", round(cv_auc.mean(), 4))

In [ ]:
# ============================================================
# Model 2: XGBoost
# ============================================================
# XGBoost (Chen and Guestrin, 2016)

# Set scale_pos_weight from the training class ratio to account for class imbalance
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
spw = neg / pos

xgb = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw,
    eval_metric='auc', random_state=42, n_jobs=-1,
)
xgb.fit(X_train, y_train)

proba = xgb.predict_proba(X_test)[:, 1]
print("Test AUC:", round(roc_auc_score(y_test, proba), 4))
print()
print(classification_report(y_test, xgb.predict(X_test)))

xgb_cv_auc = cross_val_score(xgb, X_train, y_train, cv=cv, scoring='roc_auc')
print("CV AUC (5-fold):", xgb_cv_auc.round(4))
print("Mean CV AUC:", round(xgb_cv_auc.mean(), 4))

# Store both models' fold scores for the later cross-model comparison
cv_results = {'Logistic Regression': cv_auc, 'XGBoost': xgb_cv_auc}

In [ ]:
# ============================================================
# Reusable Net Training Functions (for CV and Later RQ2 Seeds)
# ============================================================
# Adam optimiser (Kingma and Ba, 2015)

# Wrap the two nets so the same architecture can be retrained on demand, once per CV
# Fold here and once per seed in RQ2. One definition guarantees the fold and seed models
# Match the headline models below in shape

def train_tf_net(X_tr, y_tr, seed=42, epochs=50):
    tf.random.set_seed(seed)
    cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_tr)
    model = keras.Sequential([
        keras.layers.Input(shape=(X_tr.shape[1],)),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(16, activation='relu'),
        keras.layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy')
    model.fit(X_tr, y_tr, validation_split=0.2, epochs=epochs, batch_size=128,
              class_weight={0: cw[0], 1: cw[1]},
              callbacks=[keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)],
              verbose=0)
    return model

def train_torch_net(X_tr, y_tr, seed=42, epochs=50):
    torch.manual_seed(seed)
    cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_tr)
    w0, w1 = float(cw[0]), float(cw[1])
    # PyTorch has no validation_split, so hold out the last 20% by hand, mirroring Keras
    n_val = int(0.2 * len(X_tr))
    Xt, Xv = X_tr[:-n_val], X_tr[-n_val:]
    yt, yv = y_tr[:-n_val], y_tr[-n_val:]
    Xt_t = torch.tensor(Xt, dtype=torch.float32); yt_t = torch.tensor(yt, dtype=torch.float32).view(-1, 1)
    Xv_t = torch.tensor(Xv, dtype=torch.float32); yv_t = torch.tensor(yv, dtype=torch.float32).view(-1, 1)
    model = nn.Sequential(
        nn.Linear(X_tr.shape[1], 32), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    # Weight each sample's loss by its class, the per-sample equivalent of Keras' class_weight
    def wbce(p, t):
        w = torch.where(t == 1, torch.tensor(w1), torch.tensor(w0))
        return F.binary_cross_entropy(p, t, weight=w)
    loader = DataLoader(TensorDataset(Xt_t, yt_t), batch_size=128, shuffle=True)
    # Early stopping on val loss, patience 8, restore best weights, matching the Keras callback
    best, best_state, wait = float('inf'), None, 0
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad(); loss = wbce(model(xb), yb); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vl = wbce(model(Xv_t), yv_t).item()
        if vl < best:
            best, wait, best_state = vl, 0, copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if wait >= 8:
                break
    model.load_state_dict(best_state)
    return model

In [ ]:
# ============================================================
# Model 3: Neural Network (TensorFlow / Keras)
# ============================================================
# Adam optimiser (Kingma and Ba, 2015)

# Set random seeds to reduce avoidable stochastic variation
tf.random.set_seed(42)
np.random.seed(42)

weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weight = {0: weights[0], 1: weights[1]}

# Convert the training and test data to float arrays for the neural-network models
# Both frameworks use the same arrays to keep the RQ2 comparison closely matched
X_train_arr = X_train.astype('float32').values
X_test_arr  = X_test.astype('float32').values
y_train_arr = y_train.astype('float32').values
y_test_arr  = y_test.astype('float32').values

# Input -> 32 -> 16 -> 1, mirrored exactly in PyTorch next; the framework comparison
# Depends on these two nets being the same shape
tf_model = keras.Sequential([
    keras.layers.Input(shape=(X_train_arr.shape[1],)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid'),
])
tf_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                 loss='binary_crossentropy', metrics=[keras.metrics.AUC(name='auc')])
history = tf_model.fit(X_train_arr, y_train_arr, validation_split=0.2, epochs=50, batch_size=128,
                       class_weight=class_weight,
                       callbacks=[keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor='val_loss')],
                       verbose=0)

proba = tf_model.predict(X_test_arr, verbose=0).ravel()
print("Test AUC:", round(roc_auc_score(y_test_arr, proba), 4))
print()
print(classification_report(y_test_arr, (proba >= 0.5).astype(int)))
print("Epochs trained:", len(history.history['loss']))

In [ ]:
# ============================================================
# Model 4: Neural Network (PyTorch)
# ============================================================

# Set random seeds to reduce avoidable stochastic variation
torch.manual_seed(42)
np.random.seed(42)

# Same last-20% validation split as the Keras model, held out by hand
n_val = int(0.2 * len(X_train_arr))
X_tr, X_val = X_train_arr[:-n_val], X_train_arr[-n_val:]
y_tr, y_val = y_train_arr[:-n_val], y_train_arr[-n_val:]

X_tr_t   = torch.tensor(X_tr,  dtype=torch.float32)
y_tr_t   = torch.tensor(y_tr,  dtype=torch.float32).view(-1, 1)
X_val_t  = torch.tensor(X_val, dtype=torch.float32)
y_val_t  = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)
X_test_t = torch.tensor(X_test_arr, dtype=torch.float32)

# Identical shape to the TF net: input -> 32 -> 16 -> 1
class Net(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 32), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        return self.net(x)

torch_model = Net(X_tr_t.shape[1])
optimizer = torch.optim.Adam(torch_model.parameters(), lr=0.001)

# Per-sample weighted loss, mirroring the Keras class_weight
w0, w1 = float(class_weight[0]), float(class_weight[1])
def weighted_bce(pred, target):
    w = torch.where(target == 1, torch.tensor(w1), torch.tensor(w0))
    return F.binary_cross_entropy(pred, target, weight=w)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=128, shuffle=True)

# Apply early stopping to validation loss with patience of 8 epochs
# and restore the model weights from the best validation epoch
best_val, best_state, patience, wait, epochs_run = float('inf'), None, 8, 0, 0
for epoch in range(50):
    torch_model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = weighted_bce(torch_model(xb), yb)
        loss.backward()
        optimizer.step()
    torch_model.eval()
    with torch.no_grad():
        val_loss = weighted_bce(torch_model(X_val_t), y_val_t).item()
    epochs_run += 1
    if val_loss < best_val:
        best_val, wait = val_loss, 0
        best_state = copy.deepcopy(torch_model.state_dict())
    else:
        wait += 1
        if wait >= patience:
            break
torch_model.load_state_dict(best_state)

torch_model.eval()
with torch.no_grad():
    proba = torch_model(X_test_t).numpy().ravel()
print("Test AUC:", round(roc_auc_score(y_test_arr, proba), 4))
print()
print(classification_report(y_test_arr, (proba >= 0.5).astype(int)))
print("Epochs trained:", epochs_run)

In [ ]:
# ============================================================
# Cross-Validate the Nets (Matched Folds for the Comparison)
# ============================================================

# Use the same fixed cross-validation folds for the neural networks so that
# all four models are evaluated on identical partitions
tf_cv_auc, torch_cv_auc = [], []
for tr_idx, va_idx in cv.split(X_train, y_train):
    Xtr = X_train.iloc[tr_idx].astype('float32').values
    ytr = y_train.iloc[tr_idx].astype('float32').values
    Xva = X_train.iloc[va_idx].astype('float32').values
    yva = y_train.iloc[va_idx].astype('float32').values

    m_tf = train_tf_net(Xtr, ytr)
    tf_cv_auc.append(roc_auc_score(yva, m_tf.predict(Xva, verbose=0).ravel()))

    m_pt = train_torch_net(Xtr, ytr)
    with torch.no_grad():
        p = m_pt(torch.tensor(Xva, dtype=torch.float32)).numpy().ravel()
    torch_cv_auc.append(roc_auc_score(yva, p))

tf_cv_auc, torch_cv_auc = np.array(tf_cv_auc), np.array(torch_cv_auc)
print("TF CV AUC:   ", tf_cv_auc.round(4), "mean", round(tf_cv_auc.mean(), 4))
print("Torch CV AUC:", torch_cv_auc.round(4), "mean", round(torch_cv_auc.mean(), 4))

# Store the neural-network fold scores alongside the logistic regression and XGBoost results
cv_results['TensorFlow NN'] = tf_cv_auc
cv_results['PyTorch NN'] = torch_cv_auc

In [ ]:
# ============================================================
# Model Comparison Summary
# ============================================================

# Per-fold AUCs for all four models: rows are folds, columns models
cv_df = pd.DataFrame(cv_results)

summary = pd.DataFrame({
    'mean_auc': cv_df.mean(),
    'std_auc':  cv_df.std(),
}).sort_values('mean_auc', ascending=False).round(4)

# The standard deviation provides additional context on the German dataset:
# mean AUCs are relatively close for several models, while the PyTorch model
# shows greater fold-to-fold variability in this run.
print(summary)
print()
print("Per-fold AUCs (this matrix is what the Friedman / Nemenyi test uses in R):")
print(cv_df.round(4))

In [ ]:
# ============================================================
# RQ3 Fairness: Four-Fifths Rule on Test Predictions
# ============================================================
# Four-fifths rule (Equal Employment Opportunity Commission, 1978); bootstrap
# Confidence intervals for the ratios are computed in R (Efron, 1979)

# Pull the protected attributes back for the test rows; kept out of scaling, so still
# On their original readable scale
demo = X_german.loc[X_test.index].copy()
demo['SEX_label'] = demo['SEX'].map({0: 'male', 1: 'female'})

# Age is banded, not raw: the four-fifths rule compares group rates and needs discrete
# Groups; these bands match the R ones
demo['AGE_band'] = pd.cut(demo['Age'], bins=[18, 25, 35, 45, 60, 100],
                          labels=['19-25', '26-35', '36-45', '46-60', '61+'])

torch_model.eval()
preds = {
    'Logistic Regression': logreg.predict(X_test),
    'XGBoost': xgb.predict(X_test),
    'TensorFlow NN': (tf_model.predict(X_test_arr, verbose=0).ravel() >= 0.5).astype(int),
    'PyTorch NN': (torch_model(torch.tensor(X_test_arr, dtype=torch.float32)).detach().numpy().ravel() >= 0.5).astype(int),
}

# Favourable outcome = predicted non-default (class 0). The ratio is the
# least-favoured group's favourable-outcome rate divided by the most-favoured
# group's rate. Values below 0.8 fall below the four-fifths screening threshold.
def four_fifths(pred, group):
    rates = pd.Series(pred == 0).groupby(group.values, observed=True).mean()
    return rates.round(3), round(rates.min() / rates.max(), 3)

# These point estimates are screening measures rather than evidence of
# discrimination. Bootstrap confidence intervals are calculated in R to show
# the uncertainty around each ratio.
for name, pred in preds.items():
    print(name)
    for attr in ['SEX_label', 'AGE_band']:
        rates, di = four_fifths(pred, demo[attr])
        flag = "   <-- below 0.8 screening threshold" if di < 0.8 else ""
        print(f"  {attr}: four-fifths ratio = {di}{flag}")
        print("   ", rates.to_dict())
    print()

In [ ]:
# ============================================================
# Export Results for the R Analysis
# ============================================================

# The tool split is deliberate: Python does modelling and explanation, R does the formal
# Testing. These CSVs are the hand-off, exported with stable column names

# Per-fold AUCs -> Friedman / Nemenyi comparison in R
cv_df.to_csv('german_cv_auc.csv', index_label='fold')

# Test predictions plus protected attributes -> four-fifths analysis and the age-gradient
# Figure in R. Column names made R-friendly (No spaces)
export = demo[['SEX', 'Age']].copy()
export['y_true'] = y_test.values
for name, pred in preds.items():
    export[name.replace(' ', '_')] = pred
export.to_csv('german_test_predictions.csv', index=False)

print("Saved german_cv_auc.csv and german_test_predictions.csv")

from google.colab import files
files.download('german_cv_auc.csv')
files.download('german_test_predictions.csv')

In [ ]:
# ============================================================
# Explanation Setup: Helpers and the German Parent-Feature Map
# ============================================================

feature_names = X_train.columns.tolist()

# One-hot spreads a categorical across several dummies, so attributions fold back to the
# Parent feature before ranking, else a split variable is unfairly penalised against a
# Numeric one. parent_of maps each encoded column to its original; numerics map to themselves
def to_parent(col):
    for c in cat_cols:
        if col == c or col.startswith(c + '_'):
            return c
    return col
parent_of = {c: to_parent(c) for c in feature_names}

# SHAP returns different shapes per explainer (List, 2-D, or 3-D with a class axis);
# Normalise to one 2-D array of the positive class so downstream code needn't special-case
def as_2d(sv):
    if isinstance(sv, list):
        sv = sv[-1]
    sv = np.asarray(sv)
    if sv.ndim == 3:
        sv = sv[:, :, -1]
    return sv

# Global importance = mean absolute attribution per feature over the explained rows,
# summed to parent features and sorted. The same aggregation procedure is applied
# consistently across explanation methods to support like-for-like comparison.
def global_importance(sv_2d):
    s = pd.Series(np.abs(sv_2d).mean(axis=0), index=feature_names)
    return s.groupby(s.index.map(parent_of)).sum().sort_values(ascending=False)

importance_store = {}

# German's test set is only 200 rows, so explain all of it (No sampling). A fixed 200-row
# Background sample is drawn once under a fixed seed for the explainers that need a reference
rng = np.random.RandomState(42)
X_explain_arr = X_test_arr
bg_idx = rng.choice(X_train_arr.shape[0], size=200, replace=False)
X_bg_arr = X_train_arr[bg_idx]
X_lime = X_explain_arr

In [ ]:
# ============================================================
# SHAP across All Four Models
# ============================================================
# Gradient explainer chosen for the networks: shares a gradient basis with
# Integrated Gradients (Sundararajan, Taly and Yan, 2017), so the two are comparable
# SHAP (Lundberg and Lee, 2017); the exact tree explainer is Lundberg et al. (2020)

# Each model gets the SHAP explainer built for its type, used at its best rather than a
# Slow model-agnostic path:
#  - LinearExplainer for logistic regression (Exact for a linear model)
#  - TreeExplainer for XGBoost (Exact, tree-specific)
#  - GradientExplainer for the two nets, over DeepExplainer because it's reliable under
#    Keras 3 and shares a gradient basis with IG, making the two directly comparable

# Background data for the Logistic Regression SHAP explainer.
# Up to 1000 training rows are supplied; on the German dataset this is all 800
# training observations. In the current SHAP implementation the masker
# internally subsamples this background to 100 observations.
background = X_train.sample(n=min(1000, len(X_train)), random_state=42)

importance_store[('Logistic Regression', 'SHAP')] = global_importance(
    as_2d(shap.LinearExplainer(logreg, background).shap_values(X_test)))
importance_store[('XGBoost', 'SHAP')] = global_importance(
    as_2d(shap.TreeExplainer(xgb).shap_values(X_test)))
importance_store[('TensorFlow NN', 'SHAP')] = global_importance(
    as_2d(shap.GradientExplainer(tf_model, X_bg_arr).shap_values(X_explain_arr)))

torch_model.eval()
importance_store[('PyTorch NN', 'SHAP')] = global_importance(as_2d(
    shap.GradientExplainer(torch_model, torch.tensor(X_bg_arr, dtype=torch.float32))
    .shap_values(torch.tensor(X_explain_arr, dtype=torch.float32))))
print("shap done")

In [ ]:
# ============================================================
# Integrated Gradients on the Two Nets (Reference Baseline)
# ============================================================
# Integrated Gradients (Sundararajan, Taly and Yan, 2017)
# The zero vector represents the training mean for standardised numeric features
# and the inactive/reference level for encoded categorical features

# IG needs a differentiable model, so it applies to the two nets only, not the linear or
# Tree models. That's why the three-way concordance in RQ1 exists for the nets alone

# PyTorch side: Captum's ready-made implementation
torch_model.eval()
X_explain_t = torch.tensor(X_explain_arr, dtype=torch.float32)
ig_attr = IntegratedGradients(torch_model).attribute(
    X_explain_t, baselines=torch.zeros_like(X_explain_t), n_steps=64)
importance_store[('PyTorch NN', 'IG')] = global_importance(ig_attr.detach().numpy())

# TensorFlow side: hand-rolled Riemann-sum IG, written out rather than pulled from a
# Library to avoid a fragile Keras dependency. Integrates the gradients along the straight
# Path from a zero baseline to each input
def tf_integrated_gradients(model, inputs, steps=64):
    inputs = tf.convert_to_tensor(inputs, dtype=tf.float32)
    baseline = tf.zeros_like(inputs)
    total = tf.zeros_like(inputs)
    for a in tf.linspace(0.0, 1.0, steps + 1):
        x = baseline + a * (inputs - baseline)
        with tf.GradientTape() as tape:
            tape.watch(x)
            preds = model(x, training=False)
        total += tape.gradient(preds, x)
    # Average gradient along the path, times the input-minus-baseline displacement
    return ((inputs - baseline) * (total / tf.cast(steps + 1, tf.float32))).numpy()

# Same zero baseline for both nets, so the RQ2 framework comparison isn't confounded by
# A different reference point on each side
importance_store[('TensorFlow NN', 'IG')] = global_importance(
    tf_integrated_gradients(tf_model, X_explain_arr))
print("ig done")

In [ ]:
# ============================================================
# LIME on All Four Models
# ============================================================
# LIME (Ribeiro, Singh and Guestrin, 2016)

# LIME is model-agnostic and is therefore applied to all four models
# Continuous discretisation is disabled so the surrogate uses the model input scale
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train_arr, feature_names=feature_names,
    class_names=['good', 'bad'], mode='classification',
    discretize_continuous=False, random_state=42)

# LIME wants a probability function per model; these wrap each to return the two-column
# [P(good), P(Bad)] it expects
def lr_pp(x):  return logreg.predict_proba(pd.DataFrame(x, columns=feature_names))
def xgb_pp(x): return xgb.predict_proba(pd.DataFrame(x, columns=feature_names))
def tf_pp(x):
    p = tf_model(x.astype('float32'), training=False).numpy().ravel()
    return np.column_stack([1 - p, p])
def torch_pp(x):
    with torch.no_grad():
        p = torch_model(torch.tensor(x, dtype=torch.float32)).numpy().ravel()
    return np.column_stack([1 - p, p])

# LIME is local by design, so build a global importance by explaining every row and
# Averaging the absolute weights, then fold to parents like the other methods, keeping the
# Comparison like-for-like
def lime_global(predict_fn):
    acc = np.zeros(len(feature_names))
    for row in X_lime:
        exp = lime_explainer.explain_instance(
            row, predict_fn, num_features=len(feature_names), labels=(1,), num_samples=1000)
        for idx, w in exp.as_map()[1]:
            acc[idx] += abs(w)
    s = pd.Series(acc / len(X_lime), index=feature_names)
    return s.groupby(s.index.map(parent_of)).sum().sort_values(ascending=False)

# The slow step: one explanation per row per model, hence the progress print
for name, fn in [('Logistic Regression', lr_pp), ('XGBoost', xgb_pp),
                 ('TensorFlow NN', tf_pp), ('PyTorch NN', torch_pp)]:
    print("running lime for", name, "...")
    importance_store[(name, 'LIME')] = lime_global(fn)
print("lime done")

In [ ]:
# ============================================================
# RQ1 Concordance: Kendall's W and Pairwise Spearman
# ============================================================
# Kendall's coefficient of concordance (Kendall and Babington Smith, 1939)

# Check that every required model-method combination is present before RQ1.
# This prevents a partially rerun notebook from silently producing incomplete results.
required_cols = [
    ('Logistic Regression', 'SHAP'),
    ('Logistic Regression', 'LIME'),
    ('XGBoost', 'SHAP'),
    ('XGBoost', 'LIME'),
    ('TensorFlow NN', 'SHAP'),
    ('TensorFlow NN', 'IG'),
    ('TensorFlow NN', 'LIME'),
    ('PyTorch NN', 'SHAP'),
    ('PyTorch NN', 'IG'),
    ('PyTorch NN', 'LIME')
]

missing_cols = [c for c in required_cols if c not in importance_store]

if missing_cols:
    raise RuntimeError(
        "RQ1 cannot run because importance_store is missing: "
        + ", ".join(map(str, missing_cols))
    )

print("RQ1 importance_store check passed: all required results are present.")

# Assemble every model-method importance into one frame, aligned on features.
imp_df = pd.DataFrame(importance_store).fillna(0.0)
imp_df.columns = pd.MultiIndex.from_tuples(
    imp_df.columns,
    names=['model', 'method']
)
ranks = imp_df.rank(ascending=False)

# Kendall's W across the three method rankings: 0 = no agreement, 1 = identical.
# This Python value is a descriptive in-notebook check only.
# The formal tie-corrected Kendall's W and significance test are calculated in R.

# Kendall's W across the three methods' rankings: 0 = no agreement, 1 = identical. A quick
# In-notebook check; the authoritative test with significance runs in R
def kendalls_w(rank_matrix):
    m, n = rank_matrix.shape[1], rank_matrix.shape[0]
    Rj = rank_matrix.sum(axis=1)
    S = ((Rj - Rj.mean()) ** 2).sum()
    return 12 * S / (m ** 2 * (n ** 3 - n))

# Three-way W on the nets only, since only they have all three methods
print("Kendall's W, three-way (SHAP / IG / LIME):")
for net in ['TensorFlow NN', 'PyTorch NN']:
    print(f"  {net}: {kendalls_w(ranks[[(net, m) for m in ['SHAP', 'IG', 'LIME']]]):.3f}")
print()

# Pairwise Spearman correlations show where agreement or disagreement occurs between
# individual explanation methods, providing detail that the overall Kendall's W does not.
# These values are used in the R concordance heatmap.
print("Pairwise Spearman between methods:")
for model in ['Logistic Regression', 'XGBoost', 'TensorFlow NN', 'PyTorch NN']:
    methods = [m for (mod, m) in imp_df.columns if mod == model]
    for i in range(len(methods)):
        for j in range(i + 1, len(methods)):
            r, _ = spearmanr(imp_df[(model, methods[i])], imp_df[(model, methods[j])])
            print(f"  {model}: {methods[i]} vs {methods[j]} = {r:.3f}")
print()

# Export the full importance matrix -> concordance analysis and heatmap in R. Columns
# Flattened to "Model__Method", the format the R grep expects
flat = imp_df.copy()
flat.columns = [f"{mod}__{meth}" for (mod, meth) in flat.columns]
flat.to_csv('german_xai_importance.csv', index_label='feature')
print("Saved german_xai_importance.csv")
from google.colab import files
files.download('german_xai_importance.csv')

In [ ]:
# ============================================================
# RQ2 Setup: Helpers, Instrumented Training Functions, Seeds
# ============================================================
# Adam optimiser (Kingma and Ba, 2015)

DATASET = 'german'

# During the RQ2 multi-seed run, importances are retained at encoded-column level.
# German one-hot encoded columns are folded back to their original parent features
# once in the export/testing section below, before any statistical comparison.
feature_names = X_train.columns.tolist()
parent_of = {c: c for c in feature_names}

# SHAP returns different shapes per explainer; normalise to one 2-D array of the positive class
def as_2d(sv):
    if isinstance(sv, list): sv = sv[-1]
    sv = np.asarray(sv)
    if sv.ndim == 3: sv = sv[:, :, -1]
    return sv

def global_importance(sv_2d):
    s = pd.Series(np.abs(sv_2d).mean(axis=0), index=feature_names)
    return s.groupby(s.index.map(parent_of)).sum().sort_values(ascending=False)

# Instrumented trainer: same architecture and hyperparameters as the headline TF net, but
# Also returns epochs, final val loss and wall-clock time for the RQ2 training comparison.
# set_random_seed seeds Python, NumPy and TensorFlow together to improve
# repeatability of initialisation and stochastic training behaviour.
# TensorFlow operation determinism is not explicitly enabled.
def train_tf_seed(X_tr, y_tr, seed, epochs=50):
    keras.utils.set_random_seed(seed)
    cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_tr)
    model = keras.Sequential([
        keras.layers.Input(shape=(X_tr.shape[1],)),
        keras.layers.Dense(32, activation='relu'), keras.layers.Dropout(0.3),
        keras.layers.Dense(16, activation='relu'), keras.layers.Dense(1, activation='sigmoid')])
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy')
    es = keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor='val_loss')
    t0 = time.perf_counter()
    h = model.fit(X_tr, y_tr, validation_split=0.2, epochs=epochs, batch_size=128,
                  class_weight={0: cw[0], 1: cw[1]}, callbacks=[es], verbose=0)
    return model, {'epochs': len(h.history['loss']),
                   'final_val_loss': min(h.history['val_loss']),
                   'train_seconds': time.perf_counter() - t0}

# The PyTorch twin, as close as the libraries allow: same shape, optimiser and learning
# Rate, same last-20% validation split and early-stopping rule, written out by hand since
# PyTorch has no built-in callback
def train_torch_seed(X_tr, y_tr, seed, epochs=50):
    torch.manual_seed(seed)
    cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_tr)
    w0, w1 = float(cw[0]), float(cw[1])
    n_val = int(0.2 * len(X_tr))
    Xt, Xv = X_tr[:-n_val], X_tr[-n_val:]
    yt, yv = y_tr[:-n_val], y_tr[-n_val:]
    Xt_t = torch.tensor(Xt, dtype=torch.float32); yt_t = torch.tensor(yt, dtype=torch.float32).view(-1, 1)
    Xv_t = torch.tensor(Xv, dtype=torch.float32); yv_t = torch.tensor(yv, dtype=torch.float32).view(-1, 1)
    model = nn.Sequential(nn.Linear(X_tr.shape[1], 32), nn.ReLU(), nn.Dropout(0.3),
                          nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    def wbce(p, t):
        w = torch.where(t == 1, torch.tensor(w1), torch.tensor(w0))
        return F.binary_cross_entropy(p, t, weight=w)
    loader = DataLoader(TensorDataset(Xt_t, yt_t), batch_size=128, shuffle=True)
    best, best_state, wait, ep = float('inf'), None, 0, 0
    t0 = time.perf_counter()
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad(); wbce(model(xb), yb).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vl = wbce(model(Xv_t), yv_t).item()
        ep += 1
        if vl < best: best, wait, best_state = vl, 0, copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if wait >= 8: break
    model.load_state_dict(best_state)
    return model, {'epochs': ep, 'final_val_loss': best, 'train_seconds': time.perf_counter() - t0}

# Use ten seeds with fixed explanation and background samples to reduce
# variation unrelated to the repeated network training
N_SEEDS = 10
seeds = list(range(1, N_SEEDS + 1))
_rng = np.random.RandomState(0)
X_expl = X_test_arr[_rng.choice(X_test_arr.shape[0], size=min(500, X_test_arr.shape[0]), replace=False)]
bg = X_train_arr[_rng.choice(X_train_arr.shape[0], size=min(200, X_train_arr.shape[0]), replace=False)]

In [ ]:
# ============================================================
# RQ2 Multi-Seed Run: Train, Score, Explain Each Net per Seed
# ============================================================

# For each seed, train both frameworks and record the training diagnostics
# and per-feature SHAP importances. Ten matched seeds are used for the
# paired framework comparison described in the dissertation.
rows_train, shap_records = [], []
for s in seeds:
    # TensorFlow: train, score on the fixed test set, explain on the fixed sample
    m_tf, info_tf = train_tf_seed(X_train_arr, y_train_arr, s)
    auc_tf = roc_auc_score(y_test_arr, m_tf(X_test_arr, training=False).numpy().ravel())
    imp_tf = global_importance(as_2d(shap.GradientExplainer(m_tf, bg).shap_values(X_expl)))
    rows_train.append({'seed': s, 'framework': 'TensorFlow', 'test_auc': auc_tf, **info_tf})
    for feat, val in imp_tf.items():
        shap_records.append({'seed': s, 'framework': 'TensorFlow', 'feature': feat, 'importance': val})

# PyTorch: same three steps and the same fixed explanation sample are used to keep
# the comparison as closely matched as possible. Remaining differences may reflect
# framework-specific implementation, initialisation and execution behaviour.
    m_pt, info_pt = train_torch_seed(X_train_arr, y_train_arr, s)
    m_pt.eval()
    with torch.no_grad():
        auc_pt = roc_auc_score(y_test_arr, m_pt(torch.tensor(X_test_arr, dtype=torch.float32)).numpy().ravel())
    imp_pt = global_importance(as_2d(
        shap.GradientExplainer(m_pt, torch.tensor(bg, dtype=torch.float32))
        .shap_values(torch.tensor(X_expl, dtype=torch.float32))))
    rows_train.append({'seed': s, 'framework': 'PyTorch', 'test_auc': auc_pt, **info_pt})
    for feat, val in imp_pt.items():
        shap_records.append({'seed': s, 'framework': 'PyTorch', 'feature': feat, 'importance': val})

    print(f"seed {s} done")

train_df = pd.DataFrame(rows_train)
shap_df = pd.DataFrame(shap_records)
print("multi-seed run complete")

In [ ]:
# ============================================================
# RQ2 Framework Comparison and Reproducibility
# ============================================================
# Paired Wilcoxon signed-rank test (Wilcoxon, 1945), Bonferroni corrected

DATASET = 'german'  # drives the export filenames, matching the Taiwan notebook

# Map one-hot encoded variables back to their original parent features before
# statistical comparison so the analysis is conducted across the 20 original features
def to_parent_de(col):
    for c in cat_cols:
        if col == c or col.startswith(c + '_'):
            return c
    return col

shap_df['feature'] = shap_df['feature'].map(to_parent_de)
shap_df = shap_df.groupby(['seed', 'framework', 'feature'], as_index=False)['importance'].sum()

# Reshape to one row per (Feature, seed) with a column per framework, so each
# Feature's ten TF values pair with its ten PyTorch values by seed
wide = shap_df.pivot_table(index=['feature', 'seed'], columns='framework', values='importance').reset_index()
feats = wide['feature'].unique()

# Paired Wilcoxon per feature across the ten seeds: does this feature's importance
# Differ systematically between the frameworks? A degenerate case (All differences
# Zero) raises ValueError, which is treated as "no difference" (P = 1)
res = []
for f in feats:
    sub = wide[wide['feature'] == f]
    try:
        _, p = wilcoxon(sub['TensorFlow'].values, sub['PyTorch'].values)
    except ValueError:
        p = 1.0
    res.append({'feature': f, 'p_value': p})
res_df = pd.DataFrame(res)

# Bonferroni across features, since one test is run per feature
res_df['p_bonferroni'] = (res_df['p_value'] * len(feats)).clip(upper=1.0)
print(f"Features differing after Bonferroni: {int((res_df['p_bonferroni'] < 0.05).sum())} of {len(feats)}")

# Export training diagnostics and per-seed SHAP -> RQ2 tests in R
train_df.to_csv(f'{DATASET}_rq2_training.csv', index=False)
shap_df.to_csv(f'{DATASET}_rq2_shap.csv', index=False)
print(f"Saved {DATASET}_rq2_training.csv and {DATASET}_rq2_shap.csv")
from google.colab import files
files.download(f'{DATASET}_rq2_training.csv')
files.download(f'{DATASET}_rq2_shap.csv')

In [ ]:
# ============================================================
# Figure: Framework Comparison (RQ2)
# ============================================================

# Put the two frameworks side by side to show the overall similarity in
# feature importance while allowing the individual differences to remain visible.
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr

means = shap_df.groupby(['feature', 'framework'])['importance'].mean().unstack()

# Eight features keeps the labels readable; these are the ones that drive the model
means['scale'] = means[['TensorFlow', 'PyTorch']].mean(axis=1)
d = means.sort_values('scale', ascending=False).head(8).sort_values('scale')

# Tidy the labels: the raw column names are long and one is fully capitalised
tidy = {
    'SEX': 'Sex',
    'Status of existing checking account': 'Checking Account Status',
    'Installment rate in percentage of disposable income': 'Instalment Rate',
    'Savings account/bonds': 'Savings Account',
    'Present employment since': 'Employment Duration',
    'Credit history': 'Credit History',
    'Credit amount': 'Credit Amount',
}
labels = [tidy.get(f, f) for f in d.index]
print(labels)

# Descriptive checks of overall attribution magnitude and feature-ranking agreement
# between the TensorFlow and PyTorch models.
print("\nTotal attribution mass:")
print(f"  TensorFlow: {means['TensorFlow'].sum():.4f}")
print(f"  PyTorch:    {means['PyTorch'].sum():.4f}")
print(f"  ratio TF/PyT: {means['TensorFlow'].sum()/means['PyTorch'].sum():.3f}")
rho, _ = spearmanr(means['TensorFlow'], means['PyTorch'])
print(f"  Spearman between rankings: {rho:.3f}")


In [ ]:
# ============================================================
# Grouped Bar Chart
# ============================================================

TEAL, RED, GREY = "#5B9C8F", "#B5556A", "#4A4A4A"

fig, ax = plt.subplots(figsize=(8.0, 4.8))
y = np.arange(len(d))
h = 0.38

ax.barh(y + h/2, d['TensorFlow'], height=h, color=TEAL,  label='TensorFlow')
ax.barh(y - h/2, d['PyTorch'],    height=h, color=RED,   label='PyTorch')

ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel("Mean absolute SHAP value", fontsize=10, color=GREY)
ax.set_ylabel("Feature", fontsize=10, color=GREY)
ax.legend(frameon=False, fontsize=9, loc='lower right')

# Gridlines only on the value axis, so bar lengths can be compared
ax.grid(axis='x', alpha=0.3, linewidth=0.6)
ax.set_axisbelow(True)
for s in ['top', 'right', 'left']:
    ax.spines[s].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Save Figure
# ============================================================

import os

# Save the figure to a local images directory
OUT_DIR = 'images'
os.makedirs(OUT_DIR, exist_ok=True)

out = os.path.join(OUT_DIR, 'fig_framework_german.png')
fig.savefig(out, dpi=220, bbox_inches='tight', facecolor='white')

print("Written:", out)

In [ ]:
# ============================================================
# RQ2: Which Features Differ, and by How Much
# ============================================================

# For features identified by the Bonferroni-corrected comparison, show the mean
# importance under each framework alongside the corrected p-value. This allows
# statistical differences to be considered together with their observed magnitude.
means = shap_df.groupby(['feature', 'framework'])['importance'].mean().unstack()
sig = res_df[res_df['p_bonferroni'] < 0.05].set_index('feature')
print(means.loc[sig.index].assign(p_bonf=sig['p_bonferroni']).round(4))

In [ ]:
# ============================================================
# References
# ============================================================

# Chen, T. and Guestrin, C. (2016) 'XGBoost: a scalable tree boosting system',
#   Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge
#   Discovery and Data Mining, pp. 785-794. doi: 10.1145/2939672.2939785.
#
# Efron, B. (1979) 'Bootstrap methods: another look at the jackknife',
#   The Annals of Statistics, 7(1), pp. 1-26.
#   doi: 10.1214/aos/1176344552.
#
# Equal Employment Opportunity Commission (1978) Uniform Guidelines on Employee
#   Selection Procedures. 29 C.F.R. Part 1607.
#
# Hofmann, H. (1994) Statlog (German Credit Data) [Dataset].
#   UCI Machine Learning Repository. doi: 10.24432/C5NC77.
#
# Kendall, M.G. and Babington Smith, B. (1939) 'The problem of m rankings',
#   The Annals of Mathematical Statistics, 10(3), pp. 275-287.
#   doi: 10.1214/aoms/1177732186.
#
# Kingma, D.P. and Ba, J. (2015) 'Adam: a method for stochastic optimization',
#   3rd International Conference on Learning Representations (ICLR).
#
# Lundberg, S.M. and Lee, S.-I. (2017) 'A unified approach to interpreting
#   model predictions', Advances in Neural Information Processing Systems,
#   30, pp. 4765-4774.
#
# Lundberg, S.M. et al. (2020) 'From local explanations to global understanding
#   with explainable AI for trees', Nature Machine Intelligence, 2(1),
#   pp. 56-67. doi: 10.1038/s42256-019-0138-9.
#
# Ribeiro, M.T., Singh, S. and Guestrin, C. (2016) '"Why should I trust you?":
#   explaining the predictions of any classifier', Proceedings of the 22nd
#   ACM SIGKDD International Conference on Knowledge Discovery and Data Mining,
#   pp. 1135-1144. doi: 10.1145/2939672.2939778.
#
# Sundararajan, M., Taly, A. and Yan, Q. (2017) 'Axiomatic attribution for deep
#   networks', Proceedings of the 34th International Conference on Machine
#   Learning, 70, pp. 3319-3328.
#
# Wilcoxon, F. (1945) 'Individual comparisons by ranking methods',
#   Biometrics Bulletin, 1(6), pp. 80-83. doi: 10.2307/3001968.
